# SCAO — NeurIPS 2026 GPU Benchmark

**Sparse Curvature-Aware Adaptive Optimizer** — reproducible benchmark for GPT-scale models.

This notebook runs the full Table 1 from the paper on a free Colab GPU.

**Recommended runtime**: GPU → T4 (free), A100 40GB (Colab Pro)

**What this measures**:
- Validation perplexity (PPL) on WikiText-2
- Wall-clock time to target PPL
- Peak GPU memory (GB)
- Training throughput (tokens/sec)

**Scales**: tiny (1M), 125m (GPT-2 small), 350m (GPT-2 medium)

## 1. Environment Setup

In [ ]:
# Verify GPU availability and specs
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'No GPU detected!')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install required packages
!pip install -q datasets tokenizers
print('Dependencies installed.')

## 2. Load SCAO Package

Upload your project zip **or** mount Google Drive. Choose one option below.

In [ ]:
# --- OPTION A: Upload ZIP from local machine ---
# Run this cell, then click 'Choose Files' to upload scao_project.zip
from google.colab import files
import zipfile, os

uploaded = files.upload()  # upload scao_project.zip
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('/content/')
        print(f'Extracted {fname}')

# Find the project root (adjust if needed)
import glob
roots = glob.glob('/content/*/scao/__init__.py')
if roots:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(roots[0]))
    print(f'Project root: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '/content'
    print('WARN: could not auto-detect project root, using /content')

In [ ]:
# --- OPTION B: Mount Google Drive ---
# (comment out Option A if using this)
from google.colab import drive
drive.mount('/content/drive')

# Edit this path to point to your project folder in Drive
PROJECT_ROOT = '/content/drive/MyDrive/BIG_TEECH'  # <-- adjust this
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Verify import works
from scao import SCAO
print(f'SCAO imported successfully from {PROJECT_ROOT}')
print(f'SCAO class: {SCAO}')

## 3. Quick Smoke Test (tiny, 1M params)

Verify everything works before running large-scale benchmarks.

In [ ]:
import os
BENCHMARK = os.path.join(PROJECT_ROOT, 'scao', 'benchmarks', 'gpt_scale_benchmark.py')

!python {BENCHMARK} \
    --scale tiny \
    --steps 100 \
    --device cuda \
    --optimizers adamw,scao \
    --csv /content/results_smoke.csv

print('\nSmoke test complete!')

## 4. Paper Table 1 — GPT-2 Small (125M)

Full benchmark matching the paper. Runtime: ~20 min on T4, ~8 min on A100.

In [ ]:
!python {BENCHMARK} \
    --scale 125m \
    --device cuda \
    --optimizers adamw,scao,diag_shampoo \
    --seeds 42,123 \
    --csv /content/results_125m.csv

print('\n125M benchmark complete!')

## 5. Paper Table 1 — GPT-2 Medium (350M)

Requires at least 16 GB VRAM. Runtime: ~60 min on A100.

In [ ]:
# Check VRAM before running
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 15:
    print(f'WARNING: Only {vram_gb:.1f} GB VRAM. 350M needs ≥16 GB. Use Colab Pro + A100.')
    print('Skipping 350M run on this runtime.')
else:
    !python {BENCHMARK} \
        --scale 350m \
        --device cuda \
        --optimizers adamw,scao \
        --seeds 42 \
        --csv /content/results_350m.csv
    print('\n350M benchmark complete!')

## 6. Results Analysis & Paper Figures

In [ ]:
import csv
import os

def load_csv(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

results_125m = load_csv('/content/results_125m.csv')
results_350m = load_csv('/content/results_350m.csv')
results_smoke = load_csv('/content/results_smoke.csv')

all_results = results_125m + results_350m + results_smoke

if all_results:
    print(f'Loaded {len(all_results)} result rows')
    # Print summary table
    print(f'\n{"Scale":<8} {"Optimizer":<16} {"PPL":>8} {"tok/s":>8} {"mem GB":>8}')
    print('-' * 55)
    for r in all_results:
        print(f'{r["scale"]:<8} {r["optimizer"]:<16} '
              f'{float(r["final_ppl"]):>8.2f} '
              f'{float(r["tokens_per_sec"]):>8.0f} '
              f'{float(r["peak_mem_gb"]):>8.3f}')
else:
    print('No results yet. Run the benchmark cells above first.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ast

def plot_convergence(results, title, save_path=None):
    """Plot PPL vs wall-clock time for each optimizer."""
    if not results:
        print(f'No data for: {title}')
        return

    COLORS = {'adamw': '#2196F3', 'scao': '#F44336', 'diag_shampoo': '#4CAF50'}
    LABELS = {'adamw': 'AdamW', 'scao': 'SCAO (ours)', 'diag_shampoo': 'Diag-Shampoo'}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Group by optimizer, average over seeds
    from collections import defaultdict
    curves = defaultdict(list)
    bars = defaultdict(list)

    for r in results:
        opt = r['optimizer']
        curve_raw = r.get('val_curve', '')
        # val_curve not in CSV — use step-level curves CSV if available
        bars[opt].append({
            'ppl': float(r['final_ppl']),
            'toks': float(r['tokens_per_sec']),
            'mem': float(r['peak_mem_gb']),
        })

    # Bar chart: final PPL
    opts = sorted(bars.keys(), key=lambda o: ['adamw', 'scao', 'diag_shampoo'].index(o)
                  if o in ['adamw', 'scao', 'diag_shampoo'] else 99)
    ppls  = [sum(v['ppl']  for v in bars[o]) / len(bars[o]) for o in opts]
    tokss = [sum(v['toks'] for v in bars[o]) / len(bars[o]) for o in opts]
    mems  = [sum(v['mem']  for v in bars[o]) / len(bars[o]) for o in opts]

    colors = [COLORS.get(o, '#9E9E9E') for o in opts]
    labels = [LABELS.get(o, o) for o in opts]

    bars1 = ax1.bar(labels, ppls, color=colors, alpha=0.85, edgecolor='black', linewidth=0.7)
    ax1.set_ylabel('Validation PPL (lower is better)', fontsize=11)
    ax1.set_title('Final Perplexity')
    ax1.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax1.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars1, ppls):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Bar chart: throughput
    bars2 = ax2.bar(labels, tokss, color=colors, alpha=0.85, edgecolor='black', linewidth=0.7)
    ax2.set_ylabel('Throughput (tokens/sec, higher is better)', fontsize=11)
    ax2.set_title('Training Throughput')
    ax2.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax2.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars2, tokss):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tokss)*0.01,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()


if results_125m:
    plot_convergence(results_125m, 'GPT-2 Small (125M) — WikiText-2',
                     save_path='/content/fig_125m.png')

if results_350m:
    plot_convergence(results_350m, 'GPT-2 Medium (350M) — WikiText-2',
                     save_path='/content/fig_350m.png')

if not results_125m and not results_350m:
    print('Run the benchmark cells first to generate results.')

In [ ]:
# Plot wall-clock convergence curves from the *_curves.csv files
import matplotlib.pyplot as plt

COLORS = {'adamw': '#2196F3', 'scao': '#F44336', 'diag_shampoo': '#4CAF50'}
LABELS = {'adamw': 'AdamW', 'scao': 'SCAO (ours)', 'diag_shampoo': 'Diag-Shampoo'}

def load_curves_csv(path):
    """Load (optimizer, scale, seed, wall_clock_s, ppl) from curves CSV."""
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

curves_125m = load_curves_csv('/content/results_125m_curves.csv')

if curves_125m:
    from collections import defaultdict
    fig, ax = plt.subplots(figsize=(9, 5))

    # Group by (optimizer, seed) → list of (t, ppl)
    groups = defaultdict(list)
    for row in curves_125m:
        key = (row['optimizer'], row['seed'])
        groups[key].append((float(row['wall_clock_s']), float(row['ppl'])))

    # Average over seeds per optimizer
    opt_curves = defaultdict(lambda: defaultdict(list))
    for (opt, seed), pts in groups.items():
        for t, ppl in pts:
            opt_curves[opt][round(t, 0)].append(ppl)

    for opt, time_dict in sorted(opt_curves.items()):
        times = sorted(time_dict.keys())
        ppls  = [sum(time_dict[t]) / len(time_dict[t]) for t in times]
        ax.plot(times, ppls,
                color=COLORS.get(opt, '#9E9E9E'),
                label=LABELS.get(opt, opt),
                linewidth=2.0)

    ax.set_xlabel('Wall-clock time (seconds)', fontsize=12)
    ax.set_ylabel('Validation PPL', fontsize=12)
    ax.set_title('GPT-2 Small (125M) — Convergence vs Wall-Clock Time', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/fig_125m_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: /content/fig_125m_curves.png')
else:
    print('curves CSV not found — run the 125M benchmark first.')

## 7. Memory Breakdown

Shows parameter memory vs optimizer state memory for each optimizer.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_memory_breakdown(results, title):
    if not results:
        print(f'No data for: {title}')
        return

    COLORS = {'adamw': '#2196F3', 'scao': '#F44336', 'diag_shampoo': '#4CAF50'}
    LABELS = {'adamw': 'AdamW', 'scao': 'SCAO (ours)', 'diag_shampoo': 'Diag-Shampoo'}

    # Average across seeds
    from collections import defaultdict
    mem_by_opt = defaultdict(list)
    for r in results:
        mem_by_opt[r['optimizer']].append(float(r['peak_mem_gb']))

    opts   = sorted(mem_by_opt, key=lambda o: ['adamw', 'scao', 'diag_shampoo'].index(o)
                    if o in ['adamw', 'scao', 'diag_shampoo'] else 99)
    labels = [LABELS.get(o, o) for o in opts]
    mems   = [sum(mem_by_opt[o]) / len(mem_by_opt[o]) for o in opts]
    colors = [COLORS.get(o, '#9E9E9E') for o in opts]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(labels, mems, color=colors, alpha=0.85, edgecolor='black', linewidth=0.7)
    ax.set_ylabel('Peak Memory (GB)', fontsize=11)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, mems):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(mems)*0.01,
                f'{val:.2f} GB', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Annotate relative to AdamW
    if 'adamw' in opts:
        adamw_mem = mems[opts.index('adamw')]
        if 'scao' in opts:
            scao_mem = mems[opts.index('scao')]
            delta_pct = (scao_mem - adamw_mem) / adamw_mem * 100
            sign = '+' if delta_pct > 0 else ''
            ax.annotate(f'SCAO: {sign}{delta_pct:.0f}% vs AdamW',
                        xy=(0.5, 0.95), xycoords='axes fraction',
                        ha='center', fontsize=10, color='gray')

    plt.tight_layout()
    plt.savefig('/content/fig_memory.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: /content/fig_memory.png')


if results_125m:
    plot_memory_breakdown(results_125m, 'Peak GPU Memory — GPT-2 Small (125M)')
elif results_smoke:
    plot_memory_breakdown(results_smoke, 'Peak GPU Memory — Tiny (1M smoke test)')

## 8. Time-to-Target PPL (Speedup Table)

In [ ]:
# Compute speedup: time for SCAO to reach the same PPL that AdamW reaches at its final step
# Uses the curves CSV for fine-grained wall-clock interpolation

import csv, os
from collections import defaultdict
import numpy as np

def time_to_ppl(curves_path, target_ppl=None):
    """Return {optimizer: seconds_to_reach_target_ppl} from a curves CSV."""
    if not os.path.exists(curves_path):
        print(f'Not found: {curves_path}')
        return {}

    with open(curves_path) as f:
        rows = list(csv.DictReader(f))

    # Group by optimizer
    by_opt = defaultdict(list)
    for r in rows:
        by_opt[r['optimizer']].append((float(r['wall_clock_s']), float(r['ppl'])))

    # Sort each curve by time
    for opt in by_opt:
        by_opt[opt].sort()

    # Auto-set target_ppl as AdamW's final PPL
    if target_ppl is None and 'adamw' in by_opt:
        target_ppl = by_opt['adamw'][-1][1]
        print(f'Target PPL (AdamW final): {target_ppl:.2f}')

    result = {}
    for opt, pts in by_opt.items():
        times = [p[0] for p in pts]
        ppls  = [p[1] for p in pts]
        # Find first time PPL <= target
        reached = [(t, p) for t, p in pts if p <= target_ppl]
        if reached:
            result[opt] = reached[0][0]
        else:
            result[opt] = None  # never reached

    return result, target_ppl


for scale, curves_csv in [('125M', '/content/results_125m_curves.csv'),
                           ('350M', '/content/results_350m_curves.csv')]:
    if not os.path.exists(curves_csv):
        continue
    ttp, target = time_to_ppl(curves_csv)
    print(f'\n=== Time-to-PPL={target:.2f} ({scale}) ===')
    print(f'  {"Optimizer":<16}  {"Time (s)":>10}  {"Speedup vs AdamW":>18}')
    print('  ' + '-' * 50)
    baseline = ttp.get('adamw')
    for opt, t in sorted(ttp.items()):
        if t is None:
            print(f'  {opt:<16}  {"never":>10}  {"—":>18}')
        else:
            speedup = baseline / t if (baseline and t > 0) else float('nan')
            print(f'  {opt:<16}  {t:>10.1f}  {speedup:>17.2f}×')

## 9. Download Results

Download all CSVs and figures to your local machine.

In [ ]:
import glob, zipfile, os
from google.colab import files

# Package everything into a zip
output_files = (
    glob.glob('/content/results_*.csv') +
    glob.glob('/content/fig_*.png')
)

if output_files:
    zip_path = '/content/scao_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in output_files:
            zf.write(f, os.path.basename(f))
            print(f'  + {os.path.basename(f)}')
    print(f'\nDownloading {zip_path}...')
    files.download(zip_path)
else:
    print('No result files found. Run benchmark cells first.')

## Reproducibility Notes

| Item | Value |
|------|-------|
| Seeds | 42, 123 |
| Dataset | WikiText-2 (HuggingFace datasets) |
| Tokenizer | Byte-level (vocab=256) |
| Batch size | 8 (125M), 4 (350M) |
| Gradient clipping | global norm 1.0 (all optimizers) |
| LR schedule | Linear warmup → Cosine decay |
| AdamW LR | 3e-4 |
| SCAO LR | 3.51e-4 (1.17× AdamW) |
| SCAO precond_freq | max(10, steps//50) |
| SCAO rho | 0.999 |
| SCAO k_max | min(256, d_model//2) |

All results are deterministic given the same seed and GPU architecture.